In [1]:
import argparse
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
# Load dataset
data = pd.read_csv("musicData.csv")

# Check the original shape
print("Original data shape:", data.shape)

# Display basic information
data.head()

Original data shape: (50005, 18)


,instance_id,artist_name,track_name,popularity,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,obtained_date,valence,music_genre
0,32894.0,Röyksopp,Röyksopp's Night Out,27.0,0.00468,0.652,-1.0,0.941,0.79200,A#,0.115,-5.201,Minor,0.0748,100.889,4-Apr,0.759,Electronic
1,46652.0,Thievery Corporation,The Shining Path,31.0,0.01270,0.622,218293.0,0.890,0.95000,D,0.124,-7.043,Minor,0.0300,115.00200000000001,4-Apr,0.531,Electronic
2,30097.0,Dillon Francis,Hurricane,28.0,0.00306,0.620,215613.0,0.755,0.01180,G#,0.534,-4.617,Major,0.0345,127.994,4-Apr,0.333,Electronic
3,62177.0,Dubloadz,Nitro,34.0,0.02540,0.774,166875.0,0.700,0.00253,C#,0.157,-4.498,Major,0.2390,128.014,4-Apr,0.270,Electronic
4,24907.0,What So Not,Divide & Conquer,32.0,0.00465,0.638,222369.0,0.587,0.90900,F#,0.157,-6.266,Major,0.0413,145.036,4-Apr,0.323,Electronic


In [3]:
# Check column names
print(data.columns)

Index(['instance_id', 'artist_name', 'track_name', 'popularity',
       'acousticness', 'danceability', 'duration_ms', 'energy',
       'instrumentalness', 'key', 'liveness', 'loudness', 'mode',
       'speechiness', 'tempo', 'obtained_date', 'valence', 'music_genre'],
      dtype='object')


In [4]:
# Check data types
print(data.dtypes)

instance_id         float64
artist_name          object
track_name           object
popularity          float64
acousticness        float64
danceability        float64
duration_ms         float64
energy              float64
instrumentalness    float64
key                  object
liveness            float64
loudness            float64
mode                 object
speechiness         float64
tempo                object
obtained_date        object
valence             float64
music_genre          object
dtype: object


In [5]:
# Check unique values for suspicious columns
for col in data.columns:
    if data[col].dtype == "object":
        print(f"\nColumn: {col}")
        print(data[col].head())


Column: artist_name
0                Röyksopp
1    Thievery Corporation
2          Dillon Francis
3                Dubloadz
4             What So Not
Name: artist_name, dtype: object

Column: track_name
0    Röyksopp's Night Out
1        The Shining Path
2               Hurricane
3                   Nitro
4        Divide & Conquer
Name: track_name, dtype: object

Column: key
0    A#
1     D
2    G#
3    C#
4    F#
Name: key, dtype: object

Column: mode
0    Minor
1    Minor
2    Major
3    Major
4    Major
Name: mode, dtype: object

Column: tempo
0               100.889
1    115.00200000000001
2               127.994
3               128.014
4               145.036
Name: tempo, dtype: object

Column: obtained_date
0    4-Apr
1    4-Apr
2    4-Apr
3    4-Apr
4    4-Apr
Name: obtained_date, dtype: object

Column: music_genre
0    Electronic
1    Electronic
2    Electronic
3    Electronic
4    Electronic
Name: music_genre, dtype: object


In [6]:
# Check missing values before cleaning
print("Missing values before cleaning:")
print(data.isnull().sum())

Missing values before cleaning:
instance_id         5
artist_name         5
track_name          5
popularity          5
acousticness        5
danceability        5
duration_ms         5
energy              5
instrumentalness    5
key                 5
liveness            5
loudness            5
mode                5
speechiness         5
tempo               5
obtained_date       5
valence             5
music_genre         5
dtype: int64


In [7]:
df = data.copy()

# Try converting all columns to numeric (without modifying original)
numeric_candidates = []

for col in df.columns:
    converted = pd.to_numeric(df[col], errors="coerce")
    
    # If most values are numeric → treat as numeric
    if converted.notna().sum() > 0.8 * len(df):
        numeric_candidates.append(col)

print("Detected numeric columns:")
print(numeric_candidates)

Detected numeric columns:
['instance_id', 'popularity', 'acousticness', 'danceability', 'duration_ms', 'energy', 'instrumentalness', 'liveness', 'loudness', 'speechiness', 'tempo', 'valence']


In [8]:
for col in numeric_candidates:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [9]:
# Remove duplicate rows
data = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (50005, 18)


In [10]:
# Remove rows where the target variable is missing
df = df.dropna(subset=["music_genre"])

print("Shape after removing missing target:", df.shape)

Shape after removing missing target: (50000, 18)


In [11]:
df.head()

,instance_id,artist_name,track_name,popularity,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,obtained_date,valence,music_genre
0,32894.0,Röyksopp,Röyksopp's Night Out,27.0,0.00468,0.652,-1.0,0.941,0.79200,A#,0.115,-5.201,Minor,0.0748,100.889,4-Apr,0.759,Electronic
1,46652.0,Thievery Corporation,The Shining Path,31.0,0.01270,0.622,218293.0,0.890,0.95000,D,0.124,-7.043,Minor,0.0300,115.002,4-Apr,0.531,Electronic
2,30097.0,Dillon Francis,Hurricane,28.0,0.00306,0.620,215613.0,0.755,0.01180,G#,0.534,-4.617,Major,0.0345,127.994,4-Apr,0.333,Electronic
3,62177.0,Dubloadz,Nitro,34.0,0.02540,0.774,166875.0,0.700,0.00253,C#,0.157,-4.498,Major,0.2390,128.014,4-Apr,0.270,Electronic
4,24907.0,What So Not,Divide & Conquer,32.0,0.00465,0.638,222369.0,0.587,0.90900,F#,0.157,-6.266,Major,0.0413,145.036,4-Apr,0.323,Electronic


In [12]:
# Fill the missing numeric values as median values
for col in numeric_candidates:
    df[col] = df[col].fillna(df[col].median())

In [13]:
print("Final missing values:")
print(df.isnull().sum())

print("Final shape:", df.shape)

Final missing values:
instance_id         0
artist_name         0
track_name          0
popularity          0
acousticness        0
danceability        0
duration_ms         0
energy              0
instrumentalness    0
key                 0
liveness            0
loudness            0
mode                0
speechiness         0
tempo               0
obtained_date       0
valence             0
music_genre         0
dtype: int64
Final shape: (50000, 18)


In [14]:
df.to_csv("music_cleaned.csv", index=False)